# Representative false positive detections

Plots the highest-scoring **false positives** of MARS-S2L on the `test_2023` split: images with no
annotated plume that the model scores above the operating threshold of 0.5.

One figure is written per example, named after its `id_loc_image`, so that a few can be picked and
stacked into a single supplementary figure. Each figure has three panels:

| Panel | Content |
|---|---|
| RGB | current acquisition, with the **country** as the y-axis label |
| $\Delta$XCH$_4$ | MBMP retrieval in ppb, with the ERA5 **wind vector** overlaid |
| Prediction | model probability over the RGB, titled with the **scene-level score** |

The scene-level scores come from the same cached prediction CSVs used for Figures 2 and 3
([`eval_model_and_figure_prob_vs_emission_rate.ipynb`](./eval_model_and_figure_prob_vs_emission_rate.ipynb)),
so the operating point here is identical to the one reported in the paper. Onshore and offshore
predictions are combined exactly as in that notebook.

Data and model weights are read from the [Hugging Face repository](https://huggingface.co/datasets/UNEP-IMEO/MARS-S2L).

In [ ]:
%%time
import json
import os
from collections import OrderedDict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_file_system

from marss2l import dataframe_image_plumes, models, plot, validation_utils
from marss2l.huggingface import REPO_ID
from marss2l.loaders import DatasetPlumes
from marss2l.mars_sentinel2 import plume_detection, wind
from marss2l.utils import pathjoin, setup_stream_logger

fs = hf_file_system.HfFileSystem()
logger = setup_stream_logger()

SPLIT = "test_2023"
THRESHOLD_PREDICTION = 0.5
N_EXAMPLES = 24  # 20-30 examples; a handful of these go into the supplementary figure

FIGURES_DIR = "figures/false_positives"
os.makedirs(FIGURES_DIR, exist_ok=True)
device = torch.device("cpu")

## 1. Image metadata

`path_prepend_data` is required here: unlike the analysis-only notebooks, this one loads the actual
imagery, so the `s2path` / `ch4path` / `cloudmaskpath` columns have to resolve against the Hugging
Face repository.

In [ ]:
%%time
csv_path = f"datasets/{REPO_ID}/validated_images_all.csv"

dataframe_data_traintest = dataframe_image_plumes.read_csv_images(
    csv_path,
    fs=fs,
    add_columns_for_analysis=True,
    split="all",
    add_case_study=True,
    add_loc_type=True,
    path_prepend_data=f"datasets/{REPO_ID}/",
)
dataframe_data_traintest.shape

## 2. Cached scene-level predictions

Same source as Figures 2 and 3. Only the two MARS-S2L models are needed; the baselines are not
plotted here.

In [ ]:
%%time
basefolder_experiments = f"datasets/{REPO_ID}/trained_models/"

expload = [
    ("MARSS2L_20250326", "MARS-S2L", "preds_test_2023th100"),
    ("MARSS2L_off_20250523", "MARS-S2L (offshore)", "preds_test_2023thr100"),
]

outs = []
for train_folder, model_name, csv_file in expload:
    output, _ = validation_utils.load_stats_and_config(
        train_folder,
        model_name,
        csv_file=csv_file,
        basefolder_experiments=basefolder_experiments,
        fs=fs,
        logger=logger,
    )
    outs.append(output)

outs = pd.concat(outs, ignore_index=True)
outs = outs.drop(["location_name", "tile"], axis=1)

preds_test = pd.merge(
    outs,
    dataframe_data_traintest[dataframe_data_traintest.split_name == SPLIT],
    on="id_loc_image",
)
# Use isplume from the image dataframe as ground truth, as in the Figure 2/3 notebook
preds_test = preds_test.drop("target", axis=1)
preds_test["scenepredcontinuous"] = preds_test["scene_pred"]
preds_test.shape

In [ ]:
# Combine onshore and offshore predictions: each image is scored by the model that serves it
# operationally. This reproduces "MARS-S2L (combined)" of the Figure 2/3 notebook.
combined = pd.concat(
    [
        preds_test[(preds_test.model_name == "MARS-S2L") & ~preds_test.offshore],
        preds_test[(preds_test.model_name == "MARS-S2L (offshore)") & preds_test.offshore],
    ],
    ignore_index=True,
)
assert combined.id_loc_image.is_unique, "an image is scored by more than one model"
print(f"{len(combined):,} images scored ({combined.isplume.sum():,} with an annotated plume)")

## 3. Select the highest-scoring false positives

A false positive is an image with **no annotated plume** that the model scores **above 0.5**. Sorting
by score descending puts the most confident mistakes first — these are the informative ones, since a
detection scoring just over the threshold is unremarkable.

In [ ]:
false_positives = combined[
    ~combined.isplume & (combined.scenepredcontinuous > THRESHOLD_PREDICTION)
].sort_values("scenepredcontinuous", ascending=False)

print(f"{len(false_positives):,} false positives at threshold {THRESHOLD_PREDICTION}")
print(
    f"false positive rate: "
    f"{len(false_positives) / (~combined.isplume).sum():.2%}"
)

selection = false_positives.head(N_EXAMPLES).copy()
selection[
    [
        "id_loc_image",
        "location_name",
        "country",
        "satellite",
        "tile_date",
        "scenepredcontinuous",
        "offshore",
    ]
]

In [ ]:
# Where the selected examples come from -- useful for picking a set that spans several artifact
# causes rather than several images of the same site.
selection.groupby(["country", "location_name"], dropna=False).agg(
    n=("id_loc_image", "count"), max_score=("scenepredcontinuous", "max")
).sort_values("n", ascending=False)

## 4. Dataset and models

In [ ]:
%%time
dataframe_image_test, _, _ = dataframe_image_plumes.load_dataframe_split(
    dataframe_or_csv_path=dataframe_data_traintest,
    dataframe_or_csv_path_plumes=None,
    dataframe_or_csv_path_sources=None,
    split=SPLIT,
    logger=logger,
    all_locs=None,
    load_plumes=False,
)

dataset = DatasetPlumes(
    mode="test",
    strprependlogs=SPLIT,
    device=device,
    image_dataframe=dataframe_image_test,
    do_simulation=False,
    logger=logger,
    analysis_mode=True,
    cache=False,
    fs=fs,
)
dataset.image_dataframe.shape

In [ ]:
%%time
def load_weights(weights_file: str, fsmodel) -> OrderedDict:
    with fsmodel.open(weights_file, "rb") as fh:
        state_dict = torch.load(fh, map_location=device)["model_state_dict"]
    return OrderedDict(
        [(k.replace("_orig_mod.module.", ""), state_dict[k]) for k in state_dict.keys()]
    )


output_dir = f"datasets/{REPO_ID}/trained_models/MARSS2L_20250326"
output_dir_offshore = f"datasets/{REPO_ID}/trained_models/MARSS2L_off_20250523"

with fs.open(pathjoin(output_dir, "config_experiment.json")) as f:
    config = json.load(f)

model = models.load_model(
    model_name=config["model"],
    in_channels=len(dataset.bands_out),
    classification_head=False,
    logger=logger,
)
model.load_state_dict(load_weights(pathjoin(output_dir, "best_epoch"), fs), strict=False)
model = model.eval()

model_offshore = models.load_model(
    model_name=config["model"],
    in_channels=len(dataset.bands_out),
    classification_head=False,
    logger=logger,
)
model_offshore.load_state_dict(
    load_weights(pathjoin(output_dir_offshore, "best_epoch"), fs), strict=False
)
model_offshore = model_offshore.eval()

inference_function = plot.inference_function_from_torch_model(model)
inference_function_offshore = plot.inference_function_from_torch_model(model_offshore)

## 5. Plot one false positive

Panels follow the same conventions as `marss2l.plot.plot_location_for_inference.plot_location`:
$\Delta$XCH$_4$ on `plasma` scaled 0-2000 ppb, the wind arrow drawn by
`marss2l.mars_sentinel2.wind.add_wind_to_plot`, and the prediction masked below the threshold and
drawn over the RGB. The connected-component filter is applied so that what is shown is what the
operational pipeline would surface.

One wrinkle to know about: `read_csv_images` overwrites `country` with `"Offshore"` for offshore
locations, so the y-axis label reads `Offshore` rather than a country name for those examples. That is
the package convention used throughout the paper's figures, so it is left as is.

In [ ]:
def plot_false_positive(row: pd.Series, threshold_prediction: float = THRESHOLD_PREDICTION):
    """Three-panel figure for a single false positive: RGB, DeltaXCH4 + wind, prediction.

    The country goes on the y-axis of the RGB panel and the scene-level score on the title of the
    prediction panel.
    """
    # find_image() rejects tile and tile_date together; (location_name, tile_date) is unique here
    item = dataset.get_sample(location_name=row.location_name, tile_date=row.tile_date)
    infer = inference_function_offshore if row.offshore else inference_function
    preds = infer(item)

    # Keep only connected components large enough to be a detection, as in the operational pipeline
    pred_binary = plume_detection.binary_connected_prediction(
        preds, threshold_prediction=threshold_prediction
    )
    preds = preds.copy()
    preds[pred_binary == 0] = 0

    fig, axs = plt.subplots(1, 3, figsize=(11, 4), tight_layout=True)
    for ax in axs:
        ax.set_xticks([])
        ax.set_yticks([])

    satellite = str(row.satellite).replace("LC0", "L")
    date = pd.to_datetime(row.tile_date).strftime("%Y-%m-%d")

    # --- RGB of the current acquisition; country on the y axis ---
    rgb = torch.permute(item["y_context_ls0_0"][(3, 2, 1), ...], (1, 2, 0))
    axs[0].imshow(rgb.clip(0, 1))
    axs[0].set_title(f"{date} {satellite}", fontsize=12)
    country = row.country if pd.notna(row.country) else "unknown"
    axs[0].set_ylabel(country, fontsize=14)

    # --- DeltaXCH4 retrieval with the wind vector ---
    im_ch4 = axs[1].imshow(item["ch4"][0], cmap="plasma", vmin=0, vmax=2000)
    wind.add_wind_to_plot(item["wind"], ax=axs[1])
    axs[1].set_title(r"$\Delta$XCH$_4$ (ppb)", fontsize=12)
    fig.colorbar(im_ch4, ax=axs[1], fraction=0.046, pad=0.04)

    # --- Model prediction over the RGB, titled with the scene-level score ---
    axs[2].imshow(rgb.clip(0, 1))
    preds_masked = np.ma.masked_where(preds < threshold_prediction, preds)
    im_pred = axs[2].imshow(preds_masked, vmin=0, vmax=1, cmap="magma")
    axs[2].set_title(f"Prediction (score {row.scenepredcontinuous:.2f})", fontsize=12)
    fig.colorbar(im_pred, ax=axs[2], fraction=0.046, pad=0.04)

    return fig, axs

In [ ]:
# Check the layout on the highest-scoring example before writing all of them
fig, _ = plot_false_positive(selection.iloc[0])
plt.show()

## 6. Write every example

Files are named after `id_loc_image`, so a figure can be traced straight back to its row of the
prediction CSV. Both PDF (for the manuscript) and PNG (for quick review) are written.

In [ ]:
%%time
written = []
for _, row in selection.iterrows():
    try:
        fig, _ = plot_false_positive(row)
    except Exception as exc:  # keep going: one unreadable image should not stop the batch
        logger.warning(f"could not plot {row.id_loc_image}: {exc}")
        continue

    stem = pathjoin(FIGURES_DIR, str(row.id_loc_image))
    fig.savefig(f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(f"{stem}.png", bbox_inches="tight", dpi=150)
    plt.close(fig)
    written.append(
        {
            "id_loc_image": str(row.id_loc_image),
            "location_name": row.location_name,
            "country": row.country,
            "satellite": row.satellite,
            "tile_date": row.tile_date,
            "score": row.scenepredcontinuous,
            "offshore": row.offshore,
        }
    )

written = pd.DataFrame(written)
written.to_csv(pathjoin(FIGURES_DIR, "false_positives_selection.csv"), index=False)
print(f"wrote {len(written)} figures to {FIGURES_DIR}")
written

## 7. Building the supplementary figure

Pick 3-5 of the files above that span different artifact causes — changes in surface reflectance
between the reference and target acquisitions, cloud shadow or thin cirrus, smoke, missing pixels
near the swath edges, and low-albedo high-noise backgrounds — and stack them vertically.

Set `IDS_FIGURE` to the chosen `id_loc_image` values and run the cell below; it re-plots those rows
into a single figure with one example per row, which is the form the supplementary figure takes.

In [ ]:
IDS_FIGURE = []  # e.g. ["<id_loc_image>", "<id_loc_image>", "<id_loc_image>"]

if IDS_FIGURE:
    rows = selection[selection.id_loc_image.astype(str).isin([str(i) for i in IDS_FIGURE])]
    missing = set(str(i) for i in IDS_FIGURE) - set(rows.id_loc_image.astype(str))
    assert not missing, f"not in the selection: {missing}"

    fig, axs = plt.subplots(
        len(rows), 3, figsize=(11, 3.6 * len(rows)), tight_layout=True, squeeze=False
    )
    for i, (_, row) in enumerate(rows.iterrows()):
        item = dataset.get_sample(location_name=row.location_name, tile_date=row.tile_date)
        infer = inference_function_offshore if row.offshore else inference_function
        preds = infer(item)
        pred_binary = plume_detection.binary_connected_prediction(
            preds, threshold_prediction=THRESHOLD_PREDICTION
        )
        preds = preds.copy()
        preds[pred_binary == 0] = 0

        for ax in axs[i]:
            ax.set_xticks([])
            ax.set_yticks([])

        rgb = torch.permute(item["y_context_ls0_0"][(3, 2, 1), ...], (1, 2, 0))
        axs[i][0].imshow(rgb.clip(0, 1))
        axs[i][0].set_ylabel(
            row.country if pd.notna(row.country) else "unknown", fontsize=14
        )
        axs[i][1].imshow(item["ch4"][0], cmap="plasma", vmin=0, vmax=2000)
        wind.add_wind_to_plot(item["wind"], ax=axs[i][1])
        axs[i][2].imshow(rgb.clip(0, 1))
        axs[i][2].imshow(
            np.ma.masked_where(preds < THRESHOLD_PREDICTION, preds),
            vmin=0,
            vmax=1,
            cmap="magma",
        )
        axs[i][2].set_title(f"score {row.scenepredcontinuous:.2f}", fontsize=12)

        if i == 0:
            axs[i][0].set_title("RGB", fontsize=13)
            axs[i][1].set_title(r"$\Delta$XCH$_4$ (ppb)", fontsize=13)

    fig.savefig("figures/false_positives.pdf", bbox_inches="tight")
    print("wrote figures/false_positives.pdf")
else:
    print("set IDS_FIGURE to the ids chosen from the selection above")